# Train the PharmaBill tablet counter (YOLOv8n / YOLO11n)
Run on Colab with a GPU runtime. Output: `counter.tflite` + `model_meta.json` for `app/src/main/assets/models/`.

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put your Roboflow YOLO export (or the app's dataset zip, split into train/val/test) at /content/datasets/pills
!mkdir -p /content/datasets
# !unzip -q /content/drive/MyDrive/pills.zip -d /content/datasets/pills

In [ ]:
%%writefile /content/pills.yaml
path: /content/datasets/pills
train: images/train
val: images/val
test: images/test
names:
  0: tablet
  1: capsule
  2: box

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # or 'yolo11n.pt'
model.train(data='/content/pills.yaml', imgsz=640, epochs=100, batch=16,
            degrees=15, hsv_v=0.4, mosaic=1.0)

In [ ]:
metrics = model.val(data='/content/pills.yaml', split='test')
print('mAP50:', metrics.box.map50)

In [ ]:
# Exact-count accuracy and MAE per photo on the test split
import glob, os, numpy as np
errs = []
for img in glob.glob('/content/datasets/pills/images/test/*'):
    lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    true = sum(1 for l in open(lbl) if l.split()[0] in ('0', '1')) if os.path.exists(lbl) else 0
    r = model.predict(img, conf=0.40, iou=0.5, verbose=False)[0]
    pred = int(sum(1 for c in r.boxes.cls.tolist() if int(c) in (0, 1)))
    errs.append(abs(pred - true))
errs = np.array(errs)
print('exact:', (errs == 0).mean(), 'MAE:', errs.mean())

In [ ]:
path = model.export(format='tflite', imgsz=640, half=True)  # or int8=True
print(path)

In [ ]:
import json
meta = {'version': 'v0.1', 'input_size': 640, 'class_names': ['tablet', 'capsule', 'box'],
        'conf_threshold': 0.40, 'iou_threshold': 0.5}
json.dump(meta, open('/content/model_meta.json', 'w'))
# Copy the exported *_float16.tflite as counter.tflite plus model_meta.json into app/src/main/assets/models/